# Grocery Store Sales Dataset 2025
**Dataset**: pratyushpuri/grocery-store-sales-dataset-in-2025-1900-record

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

df = pd.read_csv('abhinav/grocery-store-sales-2025/grocery_chain_data.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()


## 1. Data Quality & Schema

In [ ]:
print('=== Data Types ===')
print(df.dtypes)
print(f'
=== Null Counts ===')
print(df.isnull().sum())
print(f'
=== Unique Values ===')
for col in df.columns:
    print(f'{col}: {df[col].nunique()}')


## 2. Statistical Summary

In [ ]:
df.describe()

## 3. Categorical Distributions

In [ ]:
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
n = len(cat_cols)
if n > 0:
    rows = (n + 1) // 2
    fig, axes = plt.subplots(rows, 2, figsize=(16, 5 * rows))
    axes = axes.flatten() if n > 2 else [axes] if n == 1 else axes.flatten()
    for i, col in enumerate(cat_cols):
        vc = df[col].value_counts()
        if len(vc) > 20:
            vc = vc.head(15)
        vc.plot(kind='bar', ax=axes[i], color=sns.color_palette('Set2', len(vc)))
        axes[i].set_title(f'{col} ({df[col].nunique()} unique)')
        axes[i].tick_params(axis='x', rotation=45)
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)
    plt.tight_layout()
    plt.show()


## 4. Temporal Analysis

In [ ]:
date_cols = [c for c in df.columns if 'date' in c.lower() or 'time' in c.lower()]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')
    print(f'{col}: {df[col].min()} to {df[col].max()} ({(df[col].max()-df[col].min()).days} days)')

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    df[col].dt.dayofweek.value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
    axes[0].set_title('Day of Week')
    axes[0].set_xticklabels(['Mon','Tue','Wed','Thu','Fri','Sat','Sun'], rotation=0)
    
    df[col].dt.month.value_counts().sort_index().plot(kind='bar', ax=axes[1], color='coral')
    axes[1].set_title('Monthly')
    
    df.groupby(df[col].dt.date).size().plot(ax=axes[2], color='teal')
    axes[2].set_title('Daily Transaction Volume')
    axes[2].tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()


## 5. Financial Analysis

In [ ]:
price_cols = [c for c in df.columns if any(k in c.lower() for k in ['price','cost','revenue','profit','margin','total','amount','sales'])]
qty_cols = [c for c in df.columns if any(k in c.lower() for k in ['quantity','qty','units'])]
fin_cols = [c for c in price_cols + qty_cols if df[c].dtype in ['float64','int64']]

if fin_cols:
    n = len(fin_cols)
    fig, axes = plt.subplots((n+1)//2, 2, figsize=(16, 4*((n+1)//2)))
    axes = axes.flatten()
    for i, col in enumerate(fin_cols):
        axes[i].hist(df[col].dropna(), bins=40, edgecolor='black', alpha=0.7, color=sns.color_palette('husl', n)[i])
        axes[i].set_title(f'{col} (mean={df[col].mean():.2f})')
        axes[i].axvline(df[col].mean(), color='red', linestyle='--')
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)
    plt.tight_layout()
    plt.show()

for col in fin_cols:
    print(f'{col} — Mean: {df[col].mean():.2f}, Median: {df[col].median():.2f}, Total: {df[col].sum():,.2f}')


## 6. Product Performance

In [ ]:
prod_cols = [c for c in df.columns if any(k in c.lower() for k in ['product','item','sku'])]
if prod_cols and price_cols:
    pc = prod_cols[0]
    prc = [c for c in price_cols if df[c].dtype in ['float64','int64']][0] if price_cols else None
    
    if prc:
        prod_perf = df.groupby(pc).agg(
            total_revenue=(prc, 'sum'),
            avg_revenue=(prc, 'mean'),
            count=(prc, 'count')
        ).sort_values('total_revenue', ascending=False)
        
        fig, ax = plt.subplots(figsize=(14, 8))
        prod_perf.head(20)['total_revenue'].plot(kind='barh', ax=ax, color=sns.color_palette('magma', 20))
        ax.set_title(f'Top 20 Products by Total {prc}')
        ax.set_xlabel(f'Total {prc}')
        ax.invert_yaxis()
        plt.tight_layout()
        plt.show()


## 7. Store / Location Analysis

In [ ]:
store_cols = [c for c in df.columns if any(k in c.lower() for k in ['store','location','branch','city','region'])]
if store_cols:
    for col in store_cols:
        if df[col].nunique() <= 30:
            fig, ax = plt.subplots(figsize=(12, 5))
            vc = df[col].value_counts()
            vc.plot(kind='bar', ax=ax, color=sns.color_palette('viridis', len(vc)))
            ax.set_title(f'{col} Distribution')
            plt.tight_layout()
            plt.show()


## 8. Category Analysis

In [ ]:
cat_prod_cols = [c for c in df.columns if 'category' in c.lower() or 'department' in c.lower()]
if cat_prod_cols and price_cols:
    prc = [c for c in price_cols if df[c].dtype in ['float64','int64']][0]
    for col in cat_prod_cols:
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        df[col].value_counts().plot(kind='bar', ax=axes[0], color=sns.color_palette('Set2'))
        axes[0].set_title(f'{col} — Transaction Count')
        
        df.groupby(col)[prc].sum().sort_values().plot(kind='barh', ax=axes[1], color=sns.color_palette('viridis'))
        axes[1].set_title(f'{col} — Total {prc}')
        plt.tight_layout()
        plt.show()


## 9. Correlation Matrix

In [ ]:
num_df = df.select_dtypes(include=[np.number])
if num_df.shape[1] > 2:
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(num_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=ax, center=0)
    ax.set_title('Correlation Matrix')
    plt.tight_layout()
    plt.show()


## 10. Relevance to Shelf Optimization / Planogram AI

**Strengths:**
- Price, cost, quantity data for margin-based SKU scoring
- Multi-store data for localized planograms
- Category data for shelf allocation
- Date data for seasonality detection

**Limitations:**
- Small dataset (1,900 records) — limited for ML training
- No physical shelf/aisle layout data
- No basket-level co-purchase data
